# Cumulocity → Snowflake: build up

Builds the whole setup for **one tenant**. Every object name ends with the tenant ID (e.g. `_T15264971`), so tenants don't collide. Run top to bottom; each section can be re-run on its own.

| Object | Name | Purpose |
|---|---|---|
| Catalog integration | `POLARIS_CUMULOCITY_INTEGRATION_<T>` | Connection to the Cumulocity Polaris catalog |
| Database (raw) | `CUMULOCITY_ICEBERG_<T>` | Catalog-linked, read-only Iceberg tables |
| Database (app) | `CUMULOCITY_<T>` | Everything derived from raw (raw is read-only) |
| ↳ schema `CURATED` | `DEVICE_LATEST`, `ALARM_LATEST`, `SERIES_VALUES`, `DEVICE_SERIES` | What users query |
| ↳ schema `OPS` | Procedures, `SCHEMA_STATE`, task, alert | Maintenance, not for users |
| Warehouse | `CUMULOCITY_WH_<T>` | Notebook queries and dynamic-table refreshes |
| E-mail integration | `CUMULOCITY_ALERT_EMAIL_<T>` | Optional refresh-failure alerts |

| Section | Does |
|---|---|
| 1. Parameters | Tenant values; derives all object names |
| 2. Session & Iceberg connection | Warehouse, catalog integration, raw database |
| 3. Wait for sync | Status checks |
| 4. App database | Database, schemas, refresh-failure logging |
| 5. Dynamic tables | Tables, procedures, schema-change task |
| 6. Alerting (optional) | E-mail integration, alert |
| 7. Verification | Checks and end-to-end queries |

Tear down with `cumulocity_tear_down.ipynb` (same parameters).

## 1. Parameters

Tenant connection values; all object names are derived from `catalog_name` (the Cumulocity tenant ID). The OAuth secret is stored here in plain text: a catalog integration only accepts a literal secret, and `SECRET` objects would need an external access integration, which this (trial) account doesn't allow. Don't share this notebook.

In [ ]:
import re

# Cumulocity Iceflow / Polaris tenant find docs here https://cumulocity.com/docs/datahub/streaming-lake-ingestion/#overview
catalog_host        = ""   # e.g. iceberg.cumulocity.com
catalog_name        = ""   # Polaris catalog = Cumulocity tenant ID = t15264972
oauth_client_id     = ""   # can be created in the Cumulocity via api https://cumulocity.com/docs/datahub/streaming-lake-ingestion/#obtaining-iceberg-catalog-credentials
oauth_client_secret = ""

# Alerting (section 6): must be the verified e-mail of a user in this account
alert_email = ""

# Object names, suffixed with the tenant ID
assert re.fullmatch(r"[A-Za-z0-9]+", catalog_name), "catalog_name must be alphanumeric (used in object names)"
tenant      = catalog_name.upper()
integration = f"POLARIS_CUMULOCITY_INTEGRATION_{tenant}"
raw_db      = f"CUMULOCITY_ICEBERG_{tenant}"
app_db      = f"CUMULOCITY_{tenant}"
wh          = f"CUMULOCITY_WH_{tenant}"
email_int   = f"CUMULOCITY_ALERT_EMAIL_{tenant}"

# Values are inserted into SQL cells via Jinja; reject quotes so they can't break the statements
for name, value in [("catalog_host", catalog_host), ("oauth_client_id", oauth_client_id),
                    ("oauth_client_secret", oauth_client_secret), ("alert_email", alert_email)]:
    assert "'" not in value and '"' not in value, f"{name} must not contain quotes"

for label, value in [("Integration", integration), ("Raw database", raw_db), ("App database", app_db),
                     ("Warehouse", wh), ("E-mail integration", email_int)]:
    print(f"{label:20} {value}")

## 2. Session & Iceberg connection

First the session: role `ACCOUNTADMIN` and the tenant warehouse (created here so every later cell has compute).

Then the connection: the catalog integration stores the REST endpoint and OAuth client; Snowflake polls the catalog every 30 s (`REFRESH_INTERVAL_SECONDS` default). The raw database is linked to it.

`IF NOT EXISTS` on both: replacing them would break the dependent objects. To change a tenant's connection, run the tear-down notebook first.

In [ ]:
%%sql -r set_role
USE ROLE ACCOUNTADMIN

In [ ]:
%%sql -r create_warehouse
-- X-Small, pauses after 60 s idle. Used by this notebook and by the dynamic-table refreshes
-- (SYSTEM$STREAMLIT_NOTEBOOK_WH can't be used for dynamic tables)
CREATE WAREHOUSE IF NOT EXISTS {{wh}}
  WAREHOUSE_SIZE = XSMALL
  AUTO_SUSPEND = 60
  AUTO_RESUME = TRUE
  INITIALLY_SUSPENDED = TRUE
  COMMENT = 'Cumulocity tenant {{tenant}}: notebook queries and dynamic-table refreshes'

In [ ]:
%%sql -r use_warehouse
USE WAREHOUSE {{wh}}

In [ ]:
%%sql -r create_integration
CREATE CATALOG INTEGRATION IF NOT EXISTS {{integration}}
  CATALOG_SOURCE = ICEBERG_REST
  TABLE_FORMAT = ICEBERG
  REST_CONFIG = (
    CATALOG_URI = 'https://{{catalog_host}}/api/catalog'
    CATALOG_NAME = '{{catalog_name}}'
    ACCESS_DELEGATION_MODE = VENDED_CREDENTIALS
  )
  REST_AUTHENTICATION = (
    TYPE = OAUTH
    OAUTH_CLIENT_ID = '{{oauth_client_id}}'
    OAUTH_CLIENT_SECRET = '{{oauth_client_secret}}'
    OAUTH_ALLOWED_SCOPES = ('PRINCIPAL_ROLE:ALL')
    OAUTH_TOKEN_URI = 'https://{{catalog_host}}/api/catalog/v1/oauth/tokens'
  )
  ENABLED = TRUE
  COMMENT = 'Cumulocity tenant {{tenant}}'

In [ ]:
%%sql -r verify_integration
-- Expect "success" : true
SELECT SYSTEM$VERIFY_CATALOG_INTEGRATION('{{integration}}') AS result

In [ ]:
%%sql -r create_cld
-- Auto-discovers all namespaces/tables; read-only, data stays in Cumulocity storage
CREATE DATABASE IF NOT EXISTS {{raw_db}}
  LINKED_CATALOG = (
    CATALOG = '{{integration}}',
    ALLOWED_WRITE_OPERATIONS = NONE
  )
  COMMENT = 'Cumulocity tenant {{tenant}}: raw Iceberg tables (read-only)'

## 3. Wait for sync

The first sync discovers schemas and tables, then loads each table's Iceberg snapshot. Re-run these cells until the link shows no failures and the CDC tables you need (`cdc_inventory`, `cdc_alarm`, `cdc_measurement`) show snapshots. Tables without a snapshot usually have no data at the source yet.

In [ ]:
%%sql -r link_status
-- Expect executionState RUNNING and empty failureDetails
SELECT SYSTEM$CATALOG_LINK_STATUS('{{raw_db}}') AS status

In [ ]:
%%sql -r sync_by_schema
SHOW ICEBERG TABLES IN DATABASE {{raw_db}}
->> SELECT "schema_name" AS schema_name,
       COUNT(*) AS tables,
       COUNT_IF(NOT IS_NULL_VALUE(PARSE_JSON("auto_refresh_status"):currentSnapshotId)) AS with_snapshot,
       MAX(PARSE_JSON("auto_refresh_status"):lastSnapshotTime::STRING) AS latest_snapshot
    FROM $1
    GROUP BY 1
    ORDER BY 1

## 4. App database

The raw database is read-only, so everything derived from it lives in its own database:

- **`CURATED`**: the dynamic tables users query. `LOG_LEVEL = WARN` writes their refresh failures (`FAILED`, `UPSTREAM_FAILURE`) to `SNOWFLAKE.TELEMETRY.EVENTS` for the alert in section 6.
- **`OPS`**: procedures, fingerprint table, task and alert. Grant users access to `CURATED` only.

In [ ]:
%%sql -r create_database
CREATE DATABASE IF NOT EXISTS {{app_db}}
  COMMENT = 'Cumulocity tenant {{tenant}}: objects derived from {{raw_db}}'

In [ ]:
%%sql -r create_schema
CREATE SCHEMA IF NOT EXISTS {{app_db}}.CURATED COMMENT = 'Dynamic tables for querying'

In [ ]:
%%sql -r create_ops_schema
CREATE SCHEMA IF NOT EXISTS {{app_db}}.OPS COMMENT = 'Procedures, schema fingerprint, task and alert (maintenance)'

In [ ]:
%%sql -r set_log_level
ALTER SCHEMA {{app_db}}.CURATED SET LOG_LEVEL = WARN

## 5. Dynamic tables

`latest_inventory` tables can't be read in Snowflake (Iceberg equality deletes are unsupported), so the latest state is derived from the CDC change logs:

| Table | Source | One row per | Notes |
|---|---|---|---|
| `CURATED.DEVICE_LATEST` | `cdc_inventory."inventory"` | device | Deleted devices excluded; all fields in `ATTRIBUTES` |
| `CURATED.ALARM_LATEST` | `cdc_alarm."alarm"` | alarm | Current status (ACTIVE / CLEARED) |
| `CURATED.SERIES_VALUES` (view) | every `cdc_measurement` table | measurement value | Long format: `TIME` (UTC), `DEVICE_ID`, `FRAGMENT`, `SERIES`, `VALUE`, `UNIT`, `MEASUREMENT_ID` |
| `CURATED.DEVICE_SERIES` | `SERIES_VALUES` + `DEVICE_LATEST` | device and series | Lookup for dashboards: device label, unit, count, first/last time (target lag 15 min) |

`DEVICE_LATEST` and `ALARM_LATEST` refresh incrementally, at most 5 minutes behind the source. `SERIES_VALUES` is a plain view with one `UNION ALL` branch per series; `FRAGMENT` and `SERIES` are constants per branch, so a filter on them reads only the matching tables. Events (`cdc_event."event"`) are insert-only and need no table.

**Schema evolution:** a dynamic table's column list is fixed when it's created, and typed Iceberg arrays must be cast to plain `ARRAY`. So:
- `OPS.SERIES_SELECT(pattern)` generates the long-format `SELECT` over the `cdc_measurement` tables whose name matches `pattern` (`LIKE`); every `OBJECT` column with `value`/`unit` is one series.
- `OPS.CREATE_LATEST_TABLES()` reads the current columns and generates the tables and the view.
- `OPS.REBUILD_IF_SCHEMA_CHANGED()` compares a fingerprint of the source columns (`cdc_inventory`, `cdc_alarm`, all of `cdc_measurement`) with the last build and calls it only on change.
- The task `OPS.REBUILD_ON_SCHEMA_CHANGE` runs that check every 15 minutes (serverless).

The database names are filled into the procedure bodies when they're created, so each tenant has its own procedures.

In [ ]:
%%sql -r create_proc_series
CREATE OR REPLACE PROCEDURE {{app_db}}.OPS.SERIES_SELECT(TABLE_PATTERN VARCHAR)
RETURNS VARCHAR
LANGUAGE SQL
AS
$$
DECLARE
  stmt VARCHAR;
BEGIN
  -- One UNION ALL branch per series: an OBJECT column {value, unit} of a cdc_measurement table
  -- (cast to plain OBJECT first: the structured type of a unitless series has no unit field).
  -- FRAGMENT and SERIES are constants, so filtering on them prunes the other branches.
  SELECT LISTAGG(
           'SELECT CONVERT_TIMEZONE(''UTC'', "time")::TIMESTAMP_NTZ(6) AS TIME, "source" AS DEVICE_ID, '
        || '''' || REPLACE(table_name, '''', '''''') || ''' AS FRAGMENT, '
        || '''' || REPLACE(column_name, '''', '''''') || ''' AS SERIES, '
        || '("' || REPLACE(column_name, '"', '""') || '"::OBJECT):value::FLOAT AS VALUE, '
        || '("' || REPLACE(column_name, '"', '""') || '"::OBJECT):unit::VARCHAR AS UNIT, '
        || '"id" AS MEASUREMENT_ID '
        || 'FROM {{raw_db}}."cdc_measurement"."' || REPLACE(table_name, '"', '""') || '" '
        || 'WHERE ("' || REPLACE(column_name, '"', '""') || '"::OBJECT):value IS NOT NULL',
           ' UNION ALL ') WITHIN GROUP (ORDER BY table_name, column_name)
    INTO :stmt
  FROM {{raw_db}}.INFORMATION_SCHEMA.COLUMNS
  WHERE table_schema = 'cdc_measurement' AND table_name LIKE :TABLE_PATTERN AND data_type = 'OBJECT';
  RETURN stmt;
END;
$$

In [ ]:
%%sql -r create_proc_tables
CREATE OR REPLACE PROCEDURE {{app_db}}.OPS.CREATE_LATEST_TABLES()
RETURNS VARCHAR
LANGUAGE SQL
AS
$$
DECLARE
  inv_casts VARCHAR;
  alarm_casts VARCHAR;
  series_sql VARCHAR;
  stmt VARCHAR;
BEGIN
  -- Builds ' REPLACE ("col"::ARRAY AS "col", ...)' for the structured columns of one source table
  SELECT COALESCE(' REPLACE (' || LISTAGG('"' || column_name || '"::' || IFF(data_type = 'ARRAY', 'ARRAY', 'OBJECT')
                                          || ' AS "' || column_name || '"', ', ')
                                  WITHIN GROUP (ORDER BY ordinal_position) || ')', '')
    INTO :inv_casts
  FROM {{raw_db}}.INFORMATION_SCHEMA.COLUMNS
  WHERE table_schema = 'cdc_inventory' AND table_name = 'inventory'
    AND data_type IN ('ARRAY', 'OBJECT', 'MAP');

  SELECT COALESCE(' REPLACE (' || LISTAGG('"' || column_name || '"::' || IFF(data_type = 'ARRAY', 'ARRAY', 'OBJECT')
                                          || ' AS "' || column_name || '"', ', ')
                                  WITHIN GROUP (ORDER BY ordinal_position) || ')', '')
    INTO :alarm_casts
  FROM {{raw_db}}.INFORMATION_SCHEMA.COLUMNS
  WHERE table_schema = 'cdc_alarm' AND table_name = 'alarm'
    AND data_type IN ('ARRAY', 'OBJECT', 'MAP');

  -- Latest state per device; deleted devices are filtered after picking the latest row,
  -- otherwise their previous (active) state would reappear
  stmt := 'CREATE OR REPLACE DYNAMIC TABLE {{app_db}}.CURATED.DEVICE_LATEST
       TARGET_LAG = ''5 minutes''
       WAREHOUSE = {{wh}}
       REFRESH_MODE = AUTO
       COMMENT = ''Latest state per managed object from cdc_inventory; deleted objects excluded. Join on "id". All fields in ATTRIBUTES.''
     AS
     SELECT "id", "lastUpdated", "name", "type", "owner", "eventType", "creationTime",
            OBJECT_CONSTRUCT(*) AS attributes
     FROM (
       SELECT *' || inv_casts || '
       FROM {{raw_db}}."cdc_inventory"."inventory"
       QUALIFY ROW_NUMBER() OVER (PARTITION BY "id" ORDER BY "lastUpdated" DESC) = 1
     )
     WHERE "eventType" IS DISTINCT FROM ''MANAGED_OBJECT_DELETE''';
  EXECUTE IMMEDIATE :stmt;

  -- Latest state per alarm (ALARM_CREATE + ALARM_UPDATE status changes)
  stmt := 'CREATE OR REPLACE DYNAMIC TABLE {{app_db}}.CURATED.ALARM_LATEST
       TARGET_LAG = ''5 minutes''
       WAREHOUSE = {{wh}}
       REFRESH_MODE = AUTO
       COMMENT = ''Latest state per alarm from cdc_alarm (creates + status updates). Join on "source" = DEVICE_LATEST."id".''
     AS
     SELECT *' || alarm_casts || '
     FROM {{raw_db}}."cdc_alarm"."alarm"
     QUALIFY ROW_NUMBER() OVER (PARTITION BY "id" ORDER BY "lastUpdated" DESC, "eventType" DESC) = 1';
  EXECUTE IMMEDIATE :stmt;

  -- All measurement series in long format; regenerated so new fragments and series appear
  CALL {{app_db}}.OPS.SERIES_SELECT('%') INTO :series_sql;
  stmt := 'CREATE OR REPLACE VIEW {{app_db}}.CURATED.SERIES_VALUES
       COMMENT = ''All cdc_measurement series in long format (TIME in UTC), generated by OPS.SERIES_SELECT. Filter on SERIES or FRAGMENT.''
     AS ' || series_sql;
  EXECUTE IMMEDIATE :stmt;

  -- Device/series lookup for dashboards; label "<name> · <id>" because names are not unique
  stmt := 'CREATE OR REPLACE DYNAMIC TABLE {{app_db}}.CURATED.DEVICE_SERIES
       TARGET_LAG = ''15 minutes''
       WAREHOUSE = {{wh}}
       REFRESH_MODE = AUTO
       COMMENT = ''One row per device and series from SERIES_VALUES, with the device name from DEVICE_LATEST.''
     AS
     SELECT s.DEVICE_ID,
            IFF(d."name" IS NULL, s.DEVICE_ID, d."name" || '' · '' || s.DEVICE_ID) AS DEVICE,
            s.FRAGMENT, s.SERIES, s.UNIT, s.MEASUREMENTS, s.FIRST_TIME, s.LAST_TIME
     FROM (
       SELECT DEVICE_ID, FRAGMENT, SERIES, MAX(UNIT) AS UNIT, COUNT(*) AS MEASUREMENTS,
              MIN(TIME) AS FIRST_TIME, MAX(TIME) AS LAST_TIME
       FROM {{app_db}}.CURATED.SERIES_VALUES
       GROUP BY DEVICE_ID, FRAGMENT, SERIES
     ) s
     LEFT JOIN {{app_db}}.CURATED.DEVICE_LATEST d ON d."id" = s.DEVICE_ID';
  EXECUTE IMMEDIATE :stmt;

  RETURN 'DEVICE_LATEST casts:' || inv_casts || ' | ALARM_LATEST casts:' || alarm_casts;
END;
$$

In [ ]:
%%sql -r create_schema_state
CREATE TABLE IF NOT EXISTS {{app_db}}.OPS.SCHEMA_STATE (
  signature VARCHAR,
  built_at TIMESTAMP_LTZ
)

In [ ]:
%%sql -r create_proc_rebuild
CREATE OR REPLACE PROCEDURE {{app_db}}.OPS.REBUILD_IF_SCHEMA_CHANGED()
RETURNS VARCHAR
LANGUAGE SQL
AS
$$
DECLARE
  current_sig VARCHAR;
  last_sig VARCHAR;
BEGIN
  SELECT MD5(LISTAGG(table_schema || '.' || table_name || '.' || column_name || ':' || data_type, ',')
             WITHIN GROUP (ORDER BY table_schema, table_name, column_name))
    INTO :current_sig
  FROM {{raw_db}}.INFORMATION_SCHEMA.COLUMNS
  WHERE (table_schema = 'cdc_inventory' AND table_name = 'inventory')
     OR (table_schema = 'cdc_alarm' AND table_name = 'alarm')
     OR table_schema = 'cdc_measurement';

  SELECT MAX(signature) INTO :last_sig FROM {{app_db}}.OPS.SCHEMA_STATE;

  IF (current_sig = last_sig) THEN
    RETURN 'unchanged';
  END IF;

  CALL {{app_db}}.OPS.CREATE_LATEST_TABLES();
  DELETE FROM {{app_db}}.OPS.SCHEMA_STATE;
  INSERT INTO {{app_db}}.OPS.SCHEMA_STATE VALUES (:current_sig, CURRENT_TIMESTAMP());
  RETURN 'rebuilt (signature ' || current_sig || ')';
END;
$$

In [ ]:
%%sql -r initial_build
-- First run builds both tables (no stored fingerprint). Returns 'unchanged' if nothing changed since the last build.
-- To force a rebuild (e.g. after editing CREATE_LATEST_TABLES): DELETE FROM {{app_db}}.OPS.SCHEMA_STATE, then re-run.
CALL {{app_db}}.OPS.REBUILD_IF_SCHEMA_CHANGED()

In [ ]:
%%sql -r create_task
-- Serverless: no warehouse start per check; only a rebuild uses the warehouse
CREATE OR REPLACE TASK {{app_db}}.OPS.REBUILD_ON_SCHEMA_CHANGE
  SCHEDULE = '15 MINUTE'
  USER_TASK_MANAGED_INITIAL_WAREHOUSE_SIZE = 'XSMALL'
  COMMENT = 'Rebuilds the CURATED tables and view when cdc_inventory/cdc_alarm/cdc_measurement columns change'
AS
  CALL {{app_db}}.OPS.REBUILD_IF_SCHEMA_CHANGED()

In [ ]:
%%sql -r resume_task
ALTER TASK {{app_db}}.OPS.REBUILD_ON_SCHEMA_CHANGE RESUME

## 6. Alerting (optional)

**Skip this section until `alert_email` is the verified e-mail of a user in this account** (Snowsight → profile → e-mail → verify). Otherwise the integration fails with "Email recipients ... are not allowed".

The alert checks every 5 minutes for `FAILED` / `UPSTREAM_FAILURE` refresh events of any dynamic table in `CURATED`, covering the window since its last successful run (no gaps, no duplicates). An alert that fires on each new event isn't possible here: it needs change tracking on `SNOWFLAKE.TELEMETRY.EVENTS`, which can't be enabled on that system table.

In [ ]:
%%sql -r create_email_integration
CREATE NOTIFICATION INTEGRATION IF NOT EXISTS {{email_int}}
  TYPE = EMAIL
  ENABLED = TRUE
  ALLOWED_RECIPIENTS = ('{{alert_email}}')
  COMMENT = 'Cumulocity tenant {{tenant}}: dynamic-table refresh alerts'

In [ ]:
%%sql -r create_alert
CREATE OR REPLACE ALERT {{app_db}}.OPS.DT_REFRESH_FAILURE_ALERT
  WAREHOUSE = {{wh}}
  SCHEDULE = '5 MINUTE'
  IF (EXISTS (
    SELECT 1
    FROM SNOWFLAKE.TELEMETRY.EVENTS
    WHERE resource_attributes:"snow.executable.type" = 'DYNAMIC_TABLE'
      AND resource_attributes:"snow.database.name" = '{{app_db}}'
      AND resource_attributes:"snow.schema.name" = 'CURATED'
      AND record:"name" = 'refresh.status'
      AND value:state::STRING IN ('FAILED', 'UPSTREAM_FAILURE')
      AND timestamp > CONVERT_TIMEZONE('UTC', COALESCE(SNOWFLAKE.ALERT.LAST_SUCCESSFUL_SCHEDULED_TIME(),
                                                        DATEADD('minute', -5, SNOWFLAKE.ALERT.SCHEDULED_TIME())))::TIMESTAMP_NTZ
      AND timestamp <= CONVERT_TIMEZONE('UTC', SNOWFLAKE.ALERT.SCHEDULED_TIME())::TIMESTAMP_NTZ
  ))
  THEN
    BEGIN
      LET details VARCHAR;
      SELECT LISTAGG(dt || ' [' || state || '] ' || ts || ': ' || COALESCE(msg, ''), '\n')
        INTO :details
      FROM (
        SELECT resource_attributes:"snow.executable.name"::STRING AS dt,
               value:state::STRING AS state,
               TO_VARCHAR(timestamp, 'YYYY-MM-DD HH24:MI:SS') || ' UTC' AS ts,
               value:message::STRING AS msg
        FROM SNOWFLAKE.TELEMETRY.EVENTS
        WHERE resource_attributes:"snow.executable.type" = 'DYNAMIC_TABLE'
          AND resource_attributes:"snow.database.name" = '{{app_db}}'
          AND resource_attributes:"snow.schema.name" = 'CURATED'
          AND record:"name" = 'refresh.status'
          AND value:state::STRING IN ('FAILED', 'UPSTREAM_FAILURE')
          AND timestamp > CONVERT_TIMEZONE('UTC', COALESCE(SNOWFLAKE.ALERT.LAST_SUCCESSFUL_SCHEDULED_TIME(),
                                                            DATEADD('minute', -5, SNOWFLAKE.ALERT.SCHEDULED_TIME())))::TIMESTAMP_NTZ
          AND timestamp <= CONVERT_TIMEZONE('UTC', SNOWFLAKE.ALERT.SCHEDULED_TIME())::TIMESTAMP_NTZ
        ORDER BY timestamp
        LIMIT 20
      );
      CALL SYSTEM$SEND_EMAIL(
        '{{email_int}}',
        '{{alert_email}}',
        'Snowflake: Dynamic Table refresh failed in {{app_db}}.CURATED',
        :details);
    END

In [ ]:
%%sql -r resume_alert
ALTER ALERT {{app_db}}.OPS.DT_REFRESH_FAILURE_ALERT RESUME

## 7. Verification

In [ ]:
%%sql -r dt_status
-- Expect refresh_mode = INCREMENTAL and scheduling_state = ACTIVE for both
SHOW DYNAMIC TABLES IN SCHEMA {{app_db}}.CURATED
->> SELECT "name", "rows", "refresh_mode", "refresh_mode_reason", "scheduling_state", "target_lag" FROM $1

In [ ]:
%%sql -r refresh_history
SELECT name, refresh_action, state, refresh_trigger, refresh_start_time,
       DATEDIFF('ms', refresh_start_time, refresh_end_time) AS duration_ms,
       statistics:numInsertedRows::INT AS inserted,
       statistics:numDeletedRows::INT AS deleted,
       state_message
FROM TABLE({{app_db}}.INFORMATION_SCHEMA.DYNAMIC_TABLE_REFRESH_HISTORY(NAME_PREFIX => '{{app_db}}.CURATED'))
ORDER BY refresh_start_time DESC
LIMIT 20

In [ ]:
%%sql -r task_history
SELECT name, state, scheduled_time, return_value, error_message
FROM TABLE({{app_db}}.INFORMATION_SCHEMA.TASK_HISTORY(TASK_NAME => 'REBUILD_ON_SCHEMA_CHANGE'))
WHERE database_name = '{{app_db}}'
ORDER BY scheduled_time DESC
LIMIT 10

In [ ]:
%%sql -r temperature_with_inventory
-- This tenant has no c8y_Temperature series; temperatures come from bearingTemperature and inflowTemperature
WITH temperatures AS (
  SELECT 'bearingTemperature' AS sensor, "id", "source", "time",
         "bearingTemperature":value::FLOAT AS temperature,
         "bearingTemperature":unit::STRING AS unit
  FROM {{raw_db}}."cdc_measurement"."bearingTemperature"
  UNION ALL
  SELECT 'inflowTemperature', "id", "source", "time",
         "inflowTemperature":value::FLOAT,
         "inflowTemperature":unit::STRING
  FROM {{raw_db}}."cdc_measurement"."inflowTemperature"
)
SELECT t."time",
       t.sensor,
       t.temperature,
       t.unit,
       t."source"          AS device_id,
       i."name"            AS device_name,
       i."type"            AS device_type,
       i."owner"           AS device_owner,
       i."lastUpdated"     AS device_last_updated,
       t."id"              AS measurement_id
FROM temperatures t
LEFT JOIN {{app_db}}.CURATED.DEVICE_LATEST i
  ON i."id" = t."source"
ORDER BY t."time" DESC
LIMIT 100

In [ ]:
%%sql -r active_alarms
SELECT a."time", a."severity", a."type" AS alarm_type, a."text", a."count",
       d."name" AS device_name, d."type" AS device_type
FROM {{app_db}}.CURATED.ALARM_LATEST a
LEFT JOIN {{app_db}}.CURATED.DEVICE_LATEST d ON d."id" = a."source"
WHERE a."status" = 'ACTIVE'
ORDER BY a."time" DESC